![Cyber Defense Workshop — one live investigation](../images/cyber-defense-workshop-hero-v3.png)

> **Live endpoint:** `http://127.0.0.1:8080`  
This notebook connects only to the existing workshop endpoint. It does not install or run Cloudflared, handle credentials, or manage containers.

## 1 · Endpoint prerequisite

Before running the notebook, `http://127.0.0.1:8080` must already lead to one of these environments:

### Case 1 · Container is not local

An external Cloudflare Access proxy is already listening on `127.0.0.1:8080` and forwarding requests through the read-only gateway to the remote Kustainer container.

```text
Notebook KQL request
    -> existing local proxy
    -> Cloudflare Access + Tunnel
    -> read-only Kusto gateway
    -> remote Kustainer
```

### Case 2 · Container is local

The already-running local Kustainer container publishes `127.0.0.1:8080`, so the notebook connects to it directly.

```text
Notebook KQL request
    -> 127.0.0.1:8080
    -> local Kustainer
```

The notebook does not create either route. It only decides whether the Kustainer container is local and then runs KQL through the endpoint already present.

## 2 · Decide whether the container is local and create the KQL client

The first code cell starts no process and always uses `http://127.0.0.1:8080`. A safe `.show cluster` status probe sets `CONTAINER_IS_LOCAL`:

- **`True`** when the local Kustainer endpoint returns HTTP 200.
- **`False`** when the existing read-only gateway returns HTTP 403.

The cell then requires Kusto ping and `CyberDefendStudentSnapshot` visibility. If the container is not local, it also verifies the existing gateway's `/healthz` endpoint. It never creates or changes a tunnel.

In [ ]:
from __future__ import annotations

import json
import socket
import urllib.error
import urllib.parse
import urllib.request
from typing import Any

from IPython.core.getipython import get_ipython

KUSTO_BASE_URL = "http://127.0.0.1:8080"
KUSTO_DATABASE = "CyberDefendStudentSnapshot"


def local_endpoint_is_listening(base_url: str) -> bool:
    parsed = urllib.parse.urlparse(base_url)
    if parsed.scheme != "http" or not parsed.hostname or not parsed.port:
        raise ValueError("KUSTO_BASE_URL must be an HTTP URL with an explicit local port.")
    if parsed.hostname not in {"127.0.0.1", "localhost", "::1"}:
        raise ValueError("KUSTO_BASE_URL must point to a loopback address, not the public hostname.")
    try:
        with socket.create_connection((parsed.hostname, parsed.port), timeout=1):
            return True
    except OSError:
        return False


def read_json_url(path: str, timeout: int = 30) -> dict[str, Any] | list[Any]:
    request = urllib.request.Request(f"{KUSTO_BASE_URL}{path}", method="GET")
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def make_kusto_request(csl: str, endpoint: str = "query", database: str = KUSTO_DATABASE) -> urllib.request.Request:
    if endpoint not in {"query", "mgmt"}:
        raise ValueError("endpoint must be 'query' or 'mgmt'")
    payload = json.dumps({"db": database, "csl": csl}).encode("utf-8")
    return urllib.request.Request(
        f"{KUSTO_BASE_URL}/v1/rest/{endpoint}",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )


def kusto_http_status(csl: str, endpoint: str = "mgmt", database: str = KUSTO_DATABASE) -> int:
    """Return only the HTTP status, without retaining or displaying the response body."""
    request = make_kusto_request(csl, endpoint, database)
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            return response.status
    except urllib.error.HTTPError as error:
        return error.code


def kusto_request(csl: str, endpoint: str = "query", database: str = KUSTO_DATABASE) -> dict[str, Any] | list[Any]:
    request = make_kusto_request(csl, endpoint, database)
    try:
        with urllib.request.urlopen(request, timeout=250) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Kusto endpoint returned HTTP {error.code}: {detail[:500]}") from error


def primary_records(response: dict[str, Any] | list[Any]) -> list[dict[str, Any]]:
    if isinstance(response, dict) and response.get("Tables"):
        table = response["Tables"][0]
        columns = [column["ColumnName"] for column in table.get("Columns", [])]
        return [dict(zip(columns, row)) for row in table.get("Rows", [])]

    if isinstance(response, list):
        data_table = next((frame for frame in response if frame.get("FrameType") == "DataTable"), None)
        if data_table:
            columns = [column["ColumnName"] for column in data_table["TableSchema"]["Columns"]]
            return [dict(zip(columns, row)) for row in data_table.get("Rows", [])]

    return []


def kql(query: str) -> list[dict[str, Any]]:
    """Execute KQL through the existing loopback endpoint."""
    return primary_records(kusto_request(query))


def run_kql_cell(line: str, cell: str) -> list[dict[str, Any]]:
    """Execute the raw KQL body of a %%kql notebook cell."""
    del line
    query = cell.strip()
    if not query:
        raise ValueError("The KQL cell is empty.")
    records = kql(query)
    print(f"Rows returned: {len(records)}")
    return records


ipython_shell = get_ipython()
if ipython_shell is None:
    raise RuntimeError("This setup cell must run inside an IPython/Jupyter kernel.")
ipython_shell.register_magic_function(
    run_kql_cell,
    magic_kind="cell",
    magic_name="kql",
)


def check_workshop_endpoint() -> dict[str, Any]:
    if not local_endpoint_is_listening(KUSTO_BASE_URL):
        raise RuntimeError(
            f"Nothing is listening at {KUSTO_BASE_URL}. Prepare the local container or external connection before running this notebook."
        )

    route_probe_status = kusto_http_status(".show cluster")
    if route_probe_status not in {200, 403}:
        raise RuntimeError(
            f"Could not determine whether Kustainer is local: .show cluster returned HTTP {route_probe_status}."
        )

    container_is_local = route_probe_status == 200
    gateway_ready: bool | None = None
    if not container_is_local:
        gateway_health = read_json_url("/healthz")
        gateway_ready = isinstance(gateway_health, dict) and gateway_health.get("status") == "healthy"

    kusto_ping = read_json_url("/v1/rest/ping")
    databases = primary_records(kusto_request(".show databases", endpoint="mgmt"))
    database_names = {row.get("DatabaseName") for row in databases}
    kusto_ready = isinstance(kusto_ping, dict) and kusto_ping.get("ApplicationHealthState") == "Healthy"
    database_ready = KUSTO_DATABASE in database_names

    required_checks = [kusto_ready, database_ready]
    if gateway_ready is not None:
        required_checks.append(gateway_ready)
    if not all(required_checks):
        raise RuntimeError("The existing Kusto endpoint is not ready for notebook queries.")

    return {
        "container_is_local": container_is_local,
        "kusto_healthy": kusto_ready,
        "database_visible": database_ready,
        "gateway_healthy": gateway_ready,
    }


ENDPOINT_STATUS = check_workshop_endpoint()
CONTAINER_IS_LOCAL = bool(ENDPOINT_STATUS["container_is_local"])
print(f"CONTAINER_IS_LOCAL: {CONTAINER_IS_LOCAL}")
print(f"Kusto healthy: {ENDPOINT_STATUS['kusto_healthy']}")
print(f"Database visible: {ENDPOINT_STATUS['database_visible']}")
if not CONTAINER_IS_LOCAL:
    print(f"Existing gateway healthy: {ENDPOINT_STATUS['gateway_healthy']}")
print(f"Ready for raw %%kql cells: {KUSTO_DATABASE} via {KUSTO_BASE_URL}")

## 3 · Hunt the 19 cloud adversary TTPs

This lab turns the complete [pivot-driven trainee query pack](https://github.com/dcodev1702/Cyber-Defense-Workshop-ADX/blob/main/docs/ttp-hunt-queries.kql) into **19 executable, end-to-end KQL pipelines**: 3 email hunts, 10 identity hunts, and 6 application hunts.

> **Automatic investigation rhythm**  
1. Each hunt materializes its behavior-first seed records with a KQL `let` statement.  
2. An `inner` join carries the required correlation fields into the next telemetry table.  
3. A second join completes three-stage hunts without placeholders or copied values.  
4. The final projection returns `Evidence` only for complete correlation chains.  
5. Review the Markdown above each query to explain why every join key is defensible.

### Notebook conventions

- Every query cell begins with `%%kql`; everything below that line is raw KQL sent through the detected loopback endpoint.
- Query comments and analyst guidance live in Markdown, leaving each query window clean and copy-ready.
- There are **no manual `<placeholders>`**. Run one cell to execute the entire hunt.
- No query searches for a flag literal; the final evidence appears only when all required pivots match.
- Rerun the connection cell after restarting the kernel so the `%%kql` command is registered.
- The 19 hunt cells are read-only. In Cloudflare mode the gateway enforces that boundary; in direct-local mode the notebook relies on the queries themselves because Kustainer exposes its management API locally.

| Track | Hunts | Focus |
|---|---:|---|
| 📧 Email | 3 | Mail-flow manipulation and delegated mailbox access |
| 🔐 Identity | 10 | Valid accounts, tokens, roles, policy abuse, discovery, and infrastructure access |
| 🧩 Application | 6 | Consent, service principals, API execution, partner trust, and cloud collection |

**Core scenario badge:** Hunts marked **Core scenario** form the seven-stage Midnight Blizzard-inspired path. The remaining hunts are independent extensions over the same synthetic dataset.

---
# 📧 Part I · Email-based adversary tradecraft

These hunts begin with Exchange and mailbox-control evidence, then pivot into identity or application telemetry to establish who sustained the access.

**Hunts:** `EMAIL-01` through `EMAIL-03` · **Tables touched:** `OfficeActivity`, `CloudAppEvents`, `SigninLogs`, `AADServicePrincipalSignInLogs`, and `AADNonInteractiveUserSignInLogs`

### EMAIL-01 · Mailbox Email Forwarding  
**Core scenario** · TTP ID: `mailbox-email-forwarding`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1114.003 · Email Collection: Email Forwarding Rule](https://attack.mitre.org/techniques/T1114/003/) |
| **Evidence path** | `OfficeActivity` → `CloudAppEvents` → `AADServicePrincipalSignInLogs` |
| **Question** | Was external forwarding configured and sustained by a non-human application identity? |

> **Key takeaway**  
> A forwarding rule proves mail flow changed, but not whether a user or application performed and sustained the change.

#### Automatic correlation path

1. **Seed:** external forwarding changes expose mailbox `AccountId` and source `IPAddress`.
2. **First join:** those two fields select the matching cloud-app event and carry `OAuthAppId`.
3. **Final join:** `OAuthAppId` plus `IPAddress` selects the responsible service-principal sign-in and returns `Evidence`.

**Why this is defensible:** mailbox impact alone is ambiguous; matching account, source, and application identity establishes app-backed persistence.

<details><summary><strong>Supporting reference</strong></summary>

- [Exchange mail-flow rules](https://learn.microsoft.com/exchange/security-and-compliance/mail-flow-rules/mail-flow-rules)

</details>

In [ ]:
%%kql
let Seed =
    OfficeActivity
    | where Operation == "Set-Mailbox"
    | where tostring(Parameters) has "ForwardingSmtpAddress"
    | project AccountId=UserId, IPAddress=ClientIP;
let App =
    CloudAppEvents
    | where ActionType == "MailboxForwardingConfigured"
    | project AccountId, IPAddress, OAuthAppId
    | join kind=inner Seed on AccountId, IPAddress
    | project OAuthAppId, IPAddress;
AADServicePrincipalSignInLogs
| project OAuthAppId=AppId, IPAddress, Evidence=UserAgent
| join kind=inner App on OAuthAppId, IPAddress
| project Evidence

### EMAIL-02 · Malicious Inbox Rules  
**Core scenario** · TTP ID: `malicious-inbox-rules`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1070.008 · Clear Mailbox Data](https://attack.mitre.org/techniques/T1070/008/) · [T1114.003 · Email Forwarding Rule](https://attack.mitre.org/techniques/T1114/003/) |
| **Evidence path** | `OfficeActivity` → `SigninLogs` → `AADNonInteractiveUserSignInLogs` |
| **Question** | Did an interactive Exchange session create a concealment rule and continue through background token activity? |

> **Key takeaway**  
> A rule that moves or deletes security mail shows concealment. The authentication trail identifies the initiating session and proves whether access continued.

#### Automatic correlation path

1. **Seed:** suppressive inbox rules expose `UserPrincipalName` and `IPAddress`.
2. **First join:** both fields bind the rule to an Exchange sign-in and carry its `SessionId`.
3. **Final join:** `SessionId` selects the non-interactive continuation and returns `Evidence`.

**Why this is defensible:** matching both user and source reduces false attribution; the session ID then survives the transition into background access.

<details><summary><strong>Supporting reference</strong></summary>

- [Exchange `Get-InboxRule`](https://learn.microsoft.com/powershell/module/exchange/get-inboxrule)

</details>

In [ ]:
%%kql
let Seed =
    OfficeActivity
    | where Operation == "New-InboxRule"
    | where tostring(Parameters) has_any ("DeleteMessage", "MoveToFolder")
    | project UserPrincipalName=UserId, IPAddress=ClientIP;
let Auth =
    SigninLogs
    | where AppDisplayName == "Office 365 Exchange Online" and isnotempty(SessionId)
    | project UserPrincipalName, IPAddress, SessionId
    | join kind=inner Seed on UserPrincipalName, IPAddress
    | project SessionId;
AADNonInteractiveUserSignInLogs
| where isnotempty(SessionId)
| project SessionId, Evidence=AuthenticationDetails
| join kind=inner Auth on SessionId
| project Evidence

### EMAIL-03 · Delegated Mailbox Permissions  
**Extension hunt** · TTP ID: `delegated-mailbox-permissions`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1098.002 · Additional Email Delegate Permissions](https://attack.mitre.org/techniques/T1098/002/) |
| **Evidence path** | `OfficeActivity` → `CloudAppEvents` → `AADNonInteractiveUserSignInLogs` |
| **Question** | Was mailbox access granted to another identity and then exercised through a delegated session? |

> **Key takeaway**  
> Mailbox ownership and mailbox access are different facts. A permission grant can let another identity read mail without using the owner's login.

#### Automatic correlation path

1. **Seed:** mailbox permission grants extract `DelegateUpn` and source `IPAddress`.
2. **First join:** both fields identify the delegate's cloud-app event and carry `SessionId`.
3. **Final join:** `SessionId` selects the delegate's background Exchange access and returns `Evidence`.

**Why this is defensible:** the chain distinguishes the mailbox owner from the acting delegate and proves use after authorization changed.

<details><summary><strong>Supporting references</strong></summary>

- [Exchange `Add-MailboxPermission`](https://learn.microsoft.com/powershell/module/exchange/add-mailboxpermission)
- [Defender XDR `CloudAppEvents`](https://learn.microsoft.com/defender-xdr/advanced-hunting-cloudappevents-table)

</details>

In [ ]:
%%kql
let Seed =
    OfficeActivity
    | where Operation == "Add-MailboxPermission"
    | where tostring(Parameters) has_any ("FullAccess", "SendAs", "SendOnBehalf")
    | extend DelegateUpn=extract(@"User=([^;]+)", 1, tostring(Parameters))
    | project DelegateUpn, IPAddress=ClientIP;
let Session =
    CloudAppEvents
    | where ActionType == "MailboxPermissionGranted"
    | extend SessionId=tostring(SessionData.SessionId)
    | project DelegateUpn=AccountId, IPAddress, SessionId
    | join kind=inner Seed on DelegateUpn, IPAddress
    | project SessionId;
AADNonInteractiveUserSignInLogs
| project SessionId, Evidence=AuthenticationDetails
| join kind=inner Session on SessionId
| project Evidence

---
# 🔐 Part II · Identity-based adversary tradecraft

These hunts follow cloud accounts, sessions, roles, policy changes, guest trust, and administrative control from authentication into downstream activity.

**Hunts:** `IDENTITY-01` through `IDENTITY-10` · **Primary correlation keys:** `CorrelationId`, `SessionId`, `UniqueTokenIdentifier`, identity object IDs, source IPs, policy IDs, and Azure resource names

### IDENTITY-01 · Suspicious Sign-In Patterns  
**Extension hunt** · TTP ID: `suspicious-sign-in-patterns`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1078.004 · Valid Accounts: Cloud Accounts](https://attack.mitre.org/techniques/T1078/004/) |
| **Evidence path** | `SigninLogs` → `EntraIdSignInEvents` |
| **Question** | Does a risky successful sign-in from an unmanaged device retain suspicious context in Defender XDR? |

> **Key takeaway**  
> “Suspicious sign-in” is an analytic description. The underlying behavior is successful use of a valid cloud account.

#### Automatic correlation path

1. **Seed:** risky interactive sign-ins from unmanaged devices expose `CorrelationId`.
2. **Final join:** that exact ID resolves Defender XDR's enriched view and returns `Evidence`.

**Why this is defensible:** the shared correlation ID links two views of the same authentication without approximate time, user, or IP matching.

<details><summary><strong>Supporting references</strong></summary>

- [Entra sign-in activity details](https://learn.microsoft.com/entra/identity/monitoring-health/concept-sign-in-log-activity-details)
- [Defender XDR `EntraIdSignInEvents`](https://learn.microsoft.com/defender-xdr/advanced-hunting-entraidsigninevents-table)

</details>

In [ ]:
%%kql
let Seed =
    SigninLogs
    | where ResultType == "0" and IsRisky and IsInteractive
    | where tobool(parse_json(DeviceDetail).isManaged) == false
    | project CorrelationId;
EntraIdSignInEvents
| project CorrelationId, Evidence=AuthenticationProcessingDetails
| join kind=inner Seed on CorrelationId
| project Evidence

### IDENTITY-02 · Token Abuse  
**Core scenario** · TTP ID: `token-abuse`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1528 · Steal Application Access Token](https://attack.mitre.org/techniques/T1528/) · [T1550.001 · Application Access Token](https://attack.mitre.org/techniques/T1550/001/) |
| **Evidence path** | `SigninLogs` → `AADNonInteractiveUserSignInLogs` |
| **Question** | Did a risky device-code session continue through non-interactive token use without another login or MFA prompt? |

> **Key takeaway**  
> The device-code sign-in shows token issuance. `SessionId` proves later background activity belongs to the same authenticated session.

#### Automatic correlation path

1. **Seed:** successful risky interactive device-code sign-ins expose non-empty `SessionId` values.
2. **Final join:** each session ID selects its non-interactive continuation and returns `Evidence`.

**Why this is defensible:** session continuity is stronger than matching only user or IP and explains why password changes alone may not terminate token abuse.

<details><summary><strong>Supporting reference</strong></summary>

- [Microsoft identity platform access tokens](https://learn.microsoft.com/entra/identity-platform/access-tokens)

</details>

In [ ]:
%%kql
let Seed =
    SigninLogs
    | where AuthenticationProtocol == "deviceCode" and ResultType == "0"
    | where IsRisky and IsInteractive and isnotempty(SessionId)
    | project SessionId;
AADNonInteractiveUserSignInLogs
| where not(IsInteractive) and isnotempty(SessionId)
| project SessionId, Evidence=AuthenticationDetails
| join kind=inner Seed on SessionId
| project Evidence

### IDENTITY-03 · OAuth / App Consent Abuse  
**Core scenario** · TTP ID: `oauth-app-consent-abuse`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1098.003 · Additional Cloud Roles](https://attack.mitre.org/techniques/T1098/003/) · [T1550.001 · Application Access Token](https://attack.mitre.org/techniques/T1550/001/) |
| **Evidence path** | `AuditLogs` → `CloudAppEvents` → `AADServicePrincipalSignInLogs` |
| **Question** | Did broad app-only Graph consent become durable, non-human access? |

> **Key takeaway**  
> Consent is potential authorization. The meaningful question is whether the newly authorized service principal authenticated and exercised access.

#### Automatic correlation path

1. **Seed:** broad app-role grants expose `AuditCorrelationId` and `OAuthAppId`.
2. **First join:** both IDs select the exact cloud-app permission event and carry `IPAddress`.
3. **Final join:** application ID plus source IP selects the app-only sign-in and returns `Evidence`.

**Why this is defensible:** stable audit, application, and source identifiers avoid display-name ambiguity and unrelated use of the same app.

<details><summary><strong>Supporting reference</strong></summary>

- [Audit application permission grants](https://learn.microsoft.com/entra/identity/enterprise-apps/app-perms-audit-logs)

</details>

In [ ]:
%%kql
let Audit =
    AuditLogs
    | where OperationName == "Add app role assignment to service principal"
    | where tostring(AdditionalDetails) has_all ("Mail.Read", "Files.Read.All")
    | extend OAuthAppId=tostring(TargetResources[0].appId)
    | project AuditCorrelationId=tostring(CorrelationId), OAuthAppId;
let Grant =
    CloudAppEvents
    | where ActionType == "OAuthAppPermissionGranted"
    | extend AuditCorrelationId=tostring(RawEventData.AuditCorrelationId)
    | project AuditCorrelationId, OAuthAppId, IPAddress
    | join kind=inner Audit on AuditCorrelationId, OAuthAppId
    | project OAuthAppId, IPAddress;
AADServicePrincipalSignInLogs
| project OAuthAppId=AppId, IPAddress, Evidence=UserAgent
| join kind=inner Grant on OAuthAppId, IPAddress
| project Evidence

### IDENTITY-04 · Privileged Role Abuse  
**Extension hunt** · TTP ID: `privileged-role-abuse`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1098.003 · Additional Cloud Roles](https://attack.mitre.org/techniques/T1098/003/) |
| **Evidence path** | `AuditLogs` → `SigninLogs` → `AADNonInteractiveUserSignInLogs` |
| **Question** | Did a newly elevated identity establish and continue an administrative session? |

> **Key takeaway**  
> A role grant changes authorization. Authentication evidence proves that the target—not merely the administrator making the change—used the new privilege.

#### Automatic correlation path

1. **Seed:** Global Administrator assignments expose target `UserId` and initiating `IPAddress`.
2. **First join:** both fields select the target's Azure Portal authentication and carry `SessionId`.
3. **Final join:** that session ID selects elevated background activity and returns `Evidence`.

**Why this is defensible:** the chain shows role assignment, privileged authentication, and continued use as one event path.

<details><summary><strong>Supporting references</strong></summary>

- [Secure privileged accounts](https://learn.microsoft.com/entra/architecture/security-operations-privileged-accounts)
- [Entra audit activities](https://learn.microsoft.com/entra/identity/monitoring-health/reference-audit-activities)

</details>

In [ ]:
%%kql
let Role =
    AuditLogs
    | where OperationName == "Add member to role"
    | where tostring(TargetResources) has "Global Administrator"
    | extend UserId=tostring(TargetResources[0].id), IPAddress=tostring(InitiatedBy.user.ipAddress)
    | project UserId, IPAddress;
let Session =
    SigninLogs
    | where AppDisplayName == "Azure Portal"
    | project UserId, IPAddress, SessionId
    | join kind=inner Role on UserId, IPAddress
    | project SessionId;
AADNonInteractiveUserSignInLogs
| project SessionId, Evidence=AuthenticationDetails
| join kind=inner Session on SessionId
| project Evidence

### IDENTITY-05 · Service Principal Privilege Escalation  
**Extension hunt** · TTP ID: `service-principal-privilege-escalation`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1098.003 · Additional Cloud Roles](https://attack.mitre.org/techniques/T1098/003/) |
| **Evidence path** | `AuditLogs` → `AADServicePrincipalSignInLogs` → `GraphAPIAuditEvents` |
| **Question** | Did a newly elevated service principal obtain a token and exercise directory-management privilege through Graph? |

> **Key takeaway**  
> Service principals are authorization-bearing identities. A role grant expands what an application can do without user MFA.

#### Automatic correlation path

1. **Seed:** privileged app-role grants expose `ServicePrincipalId`.
2. **First join:** that object ID selects app-only sign-ins and carries `UniqueTokenIdentifier`.
3. **Final join:** the exact token selects its Graph request and returns `Evidence`.

**Why this is defensible:** token continuity proves the role was exercised rather than merely configured.

<details><summary><strong>Supporting references</strong></summary>

- [Security operations for applications](https://learn.microsoft.com/entra/architecture/security-operations-applications)
- [Audit application permissions](https://learn.microsoft.com/entra/identity/enterprise-apps/app-perms-audit-logs)

</details>

In [ ]:
%%kql
let Role =
    AuditLogs
    | where OperationName == "Add app role assignment to service principal"
    | where tostring(AdditionalDetails) has "RoleManagement.ReadWrite.Directory"
    | extend ServicePrincipalId=tostring(TargetResources[0].id)
    | project ServicePrincipalId;
let Token =
    AADServicePrincipalSignInLogs
    | project ServicePrincipalId, UniqueTokenIdentifier
    | join kind=inner Role on ServicePrincipalId
    | project UniqueTokenIdentifier;
GraphAPIAuditEvents
| project UniqueTokenIdentifier, Evidence=RequestUri
| join kind=inner Token on UniqueTokenIdentifier
| project Evidence

### IDENTITY-06 · Conditional Access Policy Abuse  
**Core scenario** · TTP ID: `conditional-access-policy-abuse`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1556.009 · Conditional Access Policies](https://attack.mitre.org/techniques/T1556/009/) |
| **Evidence path** | `AuditLogs` → `CloudAppEvents` → `SigninLogs` |
| **Question** | Was an altered Conditional Access policy actually evaluated during a subsequent sign-in? |

> **Key takeaway**  
> A policy update is a control-plane event. Its security impact is established only when authentication is evaluated under the altered policy.

#### Automatic correlation path

1. **Seed:** policy-update audits expose `AuditCorrelationId`.
2. **First join:** that ID selects the cloud-app operation and carries normalized `PolicyId`.
3. **Final join:** expanded evaluated-policy objects are matched by policy ID and return authentication `Evidence`.

**Why this is defensible:** exact operation and policy IDs connect who changed the control, which policy changed, and the resulting sign-in behavior.

<details><summary><strong>Supporting references</strong></summary>

- [Conditional Access policy concepts](https://learn.microsoft.com/entra/identity/conditional-access/concept-conditional-access-policies)
- [Microsoft Graph sign-in resource](https://learn.microsoft.com/graph/api/resources/signin)

</details>

In [ ]:
%%kql
let Audit =
    AuditLogs
    | where OperationName == "Update conditional access policy"
    | project AuditCorrelationId=tostring(CorrelationId);
let Policy =
    CloudAppEvents
    | extend AuditCorrelationId=tostring(RawEventData.AuditCorrelationId)
    | project AuditCorrelationId, PolicyId=tostring(ObjectId)
    | join kind=inner Audit on AuditCorrelationId
    | project PolicyId;
SigninLogs
| mv-expand EvaluatedPolicy=ConditionalAccessPolicies
| project PolicyId=tostring(EvaluatedPolicy.id), Evidence=AuthenticationDetails
| join kind=inner Policy on PolicyId
| project Evidence

### IDENTITY-07 · Directory Reconnaissance  
**Extension hunt** · TTP ID: `directory-reconnaissance`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1087.004 · Cloud Account Discovery](https://attack.mitre.org/techniques/T1087/004/) · [T1069.003 · Cloud Groups Discovery](https://attack.mitre.org/techniques/T1069/003/) |
| **Evidence path** | `IdentityQueryEvents` → `SigninLogs` → `GraphAPIAuditEvents` |
| **Question** | Which authenticated actor mapped privileged users, roles, and memberships across hybrid and cloud identity planes? |

> **Key takeaway**  
> Directory reads rarely appear as change audits. Identity-query and Graph telemetry reveal who enumerated privilege paths and what endpoints they queried.

#### Automatic correlation path

1. **Seed:** targeted privilege enumeration exposes event time, `UserId`, and `IPAddress`.
2. **First join:** identity and source match Graph CLI sign-ins; a five-minute constraint preserves temporal attribution and carries `UniqueTokenIdentifier`.
3. **Final join:** the token selects the exact Graph request and returns `Evidence`.

**Why this is defensible:** identity, source, application, and bounded time isolate one token from nearby benign directory activity.

<details><summary><strong>Supporting reference</strong></summary>

- [Defender XDR `IdentityQueryEvents`](https://learn.microsoft.com/defender-xdr/advanced-hunting-identityqueryevents-table)

</details>

In [ ]:
%%kql
let Recon =
    IdentityQueryEvents
    | where QueryType == "EnumerateUsers" and Query has "adminCount=1"
    | where Application == "Microsoft Graph PowerShell" and QueryTarget == "Privileged cloud accounts"
    | project ReconTime=Timestamp, UserId=AccountObjectId, IPAddress;
let Token =
    SigninLogs
    | where AppDisplayName == "Microsoft Graph Command Line Tools" and isnotempty(UniqueTokenIdentifier)
    | project SignInTime=TimeGenerated, UserId, IPAddress, UniqueTokenIdentifier
    | join kind=inner Recon on UserId, IPAddress
    | where SignInTime between (ReconTime .. ReconTime + 5m)
    | project UniqueTokenIdentifier;
GraphAPIAuditEvents
| where isnotempty(UniqueTokenIdentifier)
| project UniqueTokenIdentifier, Evidence=RequestUri
| join kind=inner Token on UniqueTokenIdentifier
| project Evidence

### IDENTITY-08 · Session Persistence / Continuous Access  
**Extension hunt** · TTP ID: `session-persistence-continuous-access`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1550.001 · Application Access Token](https://attack.mitre.org/techniques/T1550/001/) |
| **Evidence path** | `SigninLogs` → `CloudAppEvents` → `AADNonInteractiveUserSignInLogs` |
| **Question** | Did the same authenticated session continue across services without another interactive sign-in? |

> **Key takeaway**  
> Continuous access is a timeline condition. Reusing the same `SessionId` proves activity continued through a token rather than a fresh prompt.

#### Automatic correlation path

1. **Seed:** successful interactive Microsoft 365 authentications expose `SessionId`.
2. **First join:** those IDs select later cloud-app activity and retain distinct active sessions.
3. **Final join:** each surviving session selects non-interactive authentication and returns `Evidence`.

**Why this is defensible:** cross-service session continuity explains why “no new sign-in” does not mean “no attacker.”

<details><summary><strong>Supporting references</strong></summary>

- [Non-interactive sign-ins](https://learn.microsoft.com/entra/identity/monitoring-health/concept-noninteractive-sign-ins)
- [Defender XDR `CloudAppEvents`](https://learn.microsoft.com/defender-xdr/advanced-hunting-cloudappevents-table)

</details>

In [ ]:
%%kql
let Auth =
    SigninLogs
    | where ResultType == "0" and IsInteractive and AppDisplayName == "Microsoft 365"
    | project SessionId;
let Activity =
    CloudAppEvents
    | extend SessionId=tostring(SessionData.SessionId)
    | project SessionId
    | join kind=inner Auth on SessionId
    | distinct SessionId;
AADNonInteractiveUserSignInLogs
| where not(IsInteractive)
| project SessionId, Evidence=AuthenticationDetails
| join kind=inner Activity on SessionId
| project Evidence

### IDENTITY-09 · Cross-Tenant / B2B Identity Abuse  
**Extension hunt** · TTP ID: `cross-tenant-b2b-identity-abuse`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1078.004 · Valid Cloud Accounts](https://attack.mitre.org/techniques/T1078/004/) · [T1098.003 · Additional Cloud Roles](https://attack.mitre.org/techniques/T1098/003/) |
| **Evidence path** | `AuditLogs` → `SigninLogs` → `CloudAppEvents` |
| **Question** | Did an externally owned guest identity receive sensitive authorization and use it across the tenant boundary? |

> **Key takeaway**  
> A guest suffix is not proof of cross-tenant access. Home and resource tenant inequality establishes the actual trust boundary.

#### Automatic correlation path

1. **Seed:** sensitive group additions expose `GuestUserId`.
2. **First join:** that ID selects proven B2B sign-ins and carries `SessionId`.
3. **Final join:** the session ID selects external-user cloud activity and returns `Evidence`.

**Why this is defensible:** authorization, tenant inequality, and session-bound SaaS activity turn three weak clues into one defensible chain.

<details><summary><strong>Supporting reference</strong></summary>

- [Entra sign-in activity details](https://learn.microsoft.com/entra/identity/monitoring-health/concept-sign-in-log-activity-details)

</details>

In [ ]:
%%kql
let Guest =
    AuditLogs
    | where OperationName == "Add member to group"
    | where tostring(TargetResources) has_all ("Guest", "Incident Response Administrators")
    | extend GuestUserId=tostring(TargetResources[0].id)
    | project GuestUserId;
let Session =
    SigninLogs
    | where HomeTenantId != ResourceTenantId and CrossTenantAccessType == "b2bCollaboration"
    | project GuestUserId=UserId, SessionId
    | join kind=inner Guest on GuestUserId
    | project SessionId;
CloudAppEvents
| where IsExternalUser
| extend SessionId=tostring(SessionData.SessionId)
| project SessionId, Evidence=RawEventData
| join kind=inner Session on SessionId
| project Evidence

### IDENTITY-10 · Identity to Infrastructure Access  
**Extension hunt** · TTP ID: `identity-to-infrastructure-access`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1651 · Cloud Administration Command](https://attack.mitre.org/techniques/T1651/) |
| **Evidence path** | `SigninLogs` → `AzureActivity` → `DeviceProcessEvents` |
| **Question** | Did a cloud-authenticated operator use Azure Run Command to launch code inside a virtual machine? |

> **Key takeaway**  
> Azure Run Command executes through the VM agent. The operator needs no RDP or SSH session, so control-plane and endpoint evidence must meet.

#### Automatic correlation path

1. **Seed:** successful Azure Portal sign-ins expose normalized caller identity and source IP.
2. **First join:** both fields select VM Run Command operations and carry a lowercase resource key.
3. **Final join:** that key matches the endpoint's first DNS label, while the VM-agent path proves execution and returns command-line `Evidence`.

**Why this is defensible:** cloud caller and source connect Azure Resource Manager activity to the target VM; endpoint telemetry proves in-guest execution.

<details><summary><strong>Supporting references</strong></summary>

- [Run Command for Windows VMs](https://learn.microsoft.com/azure/virtual-machines/windows/run-command)
- [Defender XDR `DeviceProcessEvents`](https://learn.microsoft.com/defender-xdr/advanced-hunting-deviceprocessevents-table)

</details>

In [ ]:
%%kql
let Identity =
    SigninLogs
    | where ResultType == "0" and AppDisplayName == "Azure Portal"
    | project Caller=UserPrincipalName, CallerIpAddress=IPAddress;
let Command =
    AzureActivity
    | where OperationNameValue =~ "MICROSOFT.COMPUTE/VIRTUALMACHINES/RUNCOMMAND/ACTION"
    | project Caller, CallerIpAddress, ResourceKey=tolower(tostring(Resource))
    | join kind=inner Identity on Caller, CallerIpAddress
    | project ResourceKey;
DeviceProcessEvents
| where FolderPath has "RunCommandWindows"
| extend ResourceKey=tolower(tostring(split(DeviceName, ".")[0]))
| project ResourceKey, Evidence=ProcessCommandLine
| join kind=inner Command on ResourceKey
| project Evidence

---
# 🧩 Part III · Application-based adversary tradecraft

These hunts reconstruct authorization, app-only or delegated token use, cross-tenant trust, API execution, resource discovery, and collection from cloud storage.

**Hunts:** `APPLICATION-01` through `APPLICATION-06` · **Primary correlation keys:** service-principal IDs, application IDs, token IDs, report IDs, consent correlation IDs, session IDs, and cloud object IDs

### APPLICATION-01 · API Access Abuse  
**Core scenario** · TTP ID: `api-access-abuse`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1059.009 · Cloud API](https://attack.mitre.org/techniques/T1059/009/) · [T1550.001 · Application Access Token](https://attack.mitre.org/techniques/T1550/001/) |
| **Evidence path** | `AuditLogs` → `AADServicePrincipalSignInLogs` → `GraphAPIAuditEvents` |
| **Question** | Did a broadly consented application authenticate and issue a specific Microsoft Graph request? |

> **Key takeaway**  
> Consent grants potential access. Only service-principal token and API records prove that the application exercised those permissions.

#### Automatic correlation path

1. **Seed:** broad application consent exposes `ServicePrincipalId`.
2. **First join:** that ID selects app-only sign-ins and carries `UniqueTokenIdentifier`.
3. **Final join:** the exact token selects its Graph request and returns `Evidence`.

**Why this is defensible:** the chain reconstructs authorization → app-only authentication → resource access without conflating unrelated calls.

<details><summary><strong>Supporting reference</strong></summary>

- [Microsoft Graph authentication concepts](https://learn.microsoft.com/graph/auth/auth-concepts)

</details>

In [ ]:
%%kql
let Consent =
    AuditLogs
    | where OperationName == "Consent to application"
    | where tostring(AdditionalDetails) has_all ("Files.Read.All", "Directory.Read.All")
    | extend ServicePrincipalId=tostring(TargetResources[0].id)
    | project ServicePrincipalId;
let Token =
    AADServicePrincipalSignInLogs
    | project ServicePrincipalId, UniqueTokenIdentifier
    | join kind=inner Consent on ServicePrincipalId
    | project UniqueTokenIdentifier;
GraphAPIAuditEvents
| project UniqueTokenIdentifier, Evidence=RequestUri
| join kind=inner Token on UniqueTokenIdentifier
| project Evidence

### APPLICATION-02 · Application Impersonation / On-Behalf-Of Abuse  
**Extension hunt** · TTP ID: `application-impersonation-obo-abuse`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1550.001 · Application Access Token](https://attack.mitre.org/techniques/T1550/001/) |
| **Evidence path** | `SigninLogs` → `AADNonInteractiveUserSignInLogs` → `GraphAPIAuditEvents` |
| **Question** | Did a trusted middle tier exchange acquired user context and use the delegated token against Graph? |

> **Key takeaway**  
> On-Behalf-Of is legitimate protocol behavior. Risk appears when a confidential application uses acquired user context abusively.

#### Automatic correlation path

1. **Seed:** successful On-Behalf-Of processing exposes `SessionId`.
2. **First join:** that session selects OAuth 2.0 background exchanges and carries `UniqueTokenIdentifier`.
3. **Final join:** the exact token selects downstream Graph activity and returns `Evidence`.

**Why this is defensible:** the chain separates the human, the application acting for that human, and the API request inheriting the user's permissions.

<details><summary><strong>Supporting references</strong></summary>

- [OAuth 2.0 On-Behalf-Of flow](https://learn.microsoft.com/entra/identity-platform/v2-oauth2-on-behalf-of-flow)
- [Non-interactive sign-ins](https://learn.microsoft.com/entra/identity/monitoring-health/concept-noninteractive-sign-ins)

</details>

In [ ]:
%%kql
let Auth =
    SigninLogs
    | where ResultType == "0" and AuthenticationProcessingDetails has "On-Behalf-Of"
    | project SessionId;
let Token =
    AADNonInteractiveUserSignInLogs
    | where AuthenticationProtocol == "oAuth2"
    | project SessionId, UniqueTokenIdentifier
    | join kind=inner Auth on SessionId
    | project UniqueTokenIdentifier;
GraphAPIAuditEvents
| project UniqueTokenIdentifier, Evidence=RequestUri
| join kind=inner Token on UniqueTokenIdentifier
| project Evidence

### APPLICATION-03 · Cross-Tenant / B2B Application Impersonation  
**Extension hunt** · TTP ID: `cross-tenant-b2b-application-impersonation`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1199 · Trusted Relationship](https://attack.mitre.org/techniques/T1199/) · [T1550.001 · Application Access Token](https://attack.mitre.org/techniques/T1550/001/) |
| **Evidence path** | `AADServicePrincipalSignInLogs` → `GraphAPIAuditEvents` → `CloudAppEvents` |
| **Question** | Did a foreign-owned service principal authenticate, call an internal API, and touch SaaS data through partner trust? |

> **Key takeaway**  
> External application ownership is measurable. Compare the app-owner tenant with the resource tenant before accepting partner branding as trust.

#### Automatic correlation path

1. **Seed:** foreign-owned app sign-ins expose `AppId` and `UniqueTokenIdentifier`.
2. **First join:** both keys select the exact Graph request and carry event-specific `ReportId`.
3. **Final join:** application and report IDs select the matching SaaS action and return `Evidence`.

**Why this is defensible:** token, application, and event-specific report IDs preserve identity and event continuity across three telemetry planes.

<details><summary><strong>Supporting reference</strong></summary>

- [Cross-tenant access overview](https://learn.microsoft.com/entra/external-id/cross-tenant-access-overview)

</details>

In [ ]:
%%kql
let SignIn =
    AADServicePrincipalSignInLogs
    | where AppOwnerTenantId != AADTenantId
    | project AppId, UniqueTokenIdentifier;
let Api =
    GraphAPIAuditEvents
    | project AppId=ApplicationId, UniqueTokenIdentifier, ReportId
    | join kind=inner SignIn on AppId, UniqueTokenIdentifier
    | project AppId, ReportId;
CloudAppEvents
| project AppId=OAuthAppId, ReportId, Evidence=RawEventData
| join kind=inner Api on AppId, ReportId
| project Evidence

### APPLICATION-04 · Illicit Consent Grants  
**Extension hunt** · TTP ID: `illicit-consent-grants`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1528 · Steal Application Access Token](https://attack.mitre.org/techniques/T1528/) · [T1550.001 · Application Access Token](https://attack.mitre.org/techniques/T1550/001/) |
| **Evidence path** | `AuditLogs` → `CloudAppEvents` → `AADNonInteractiveUserSignInLogs` |
| **Question** | Did deceptive user consent become durable access through refresh-token redemption? |

> **Key takeaway**  
> This challenge models delegated user consent—not app-only administrator consent. The user authorizes scopes, and the application redeems them in the background.

#### Automatic correlation path

1. **Seed:** high-risk principal consent exposes `AuditCorrelationId`.
2. **First join:** that ID selects the enriched consent event and carries `SessionId`.
3. **Final join:** the session selects refresh-token authentication and returns `Evidence`.

**Why this is defensible:** the chain reconstructs social authorization, application receipt, and durable token use without conflating app-role assignment.

<details><summary><strong>Supporting reference</strong></summary>

- [Audit application permission grants](https://learn.microsoft.com/entra/identity/enterprise-apps/app-perms-audit-logs)

</details>

In [ ]:
%%kql
let Consent =
    AuditLogs
    | where OperationName == "Consent to application"
    | where tostring(AdditionalDetails) has_all ("Principal", "Mail.Read", "offline_access")
    | project AuditCorrelationId=tostring(CorrelationId);
let Session =
    CloudAppEvents
    | where ActionType == "OAuthAppConsentGranted"
    | extend AuditCorrelationId=tostring(RawEventData.AuditCorrelationId), SessionId=tostring(SessionData.SessionId)
    | project AuditCorrelationId, SessionId
    | join kind=inner Consent on AuditCorrelationId
    | project SessionId;
AADNonInteractiveUserSignInLogs
| where IncomingTokenType == "refreshToken"
| project SessionId, Evidence=AuthenticationDetails
| join kind=inner Session on SessionId
| project Evidence

### APPLICATION-05 · Cloud Resource Enumeration  
**Extension hunt** · TTP ID: `cloud-resource-enumeration`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1580 · Cloud Infrastructure Discovery](https://attack.mitre.org/techniques/T1580/) · [T1059.009 · Cloud API](https://attack.mitre.org/techniques/T1059/009/) |
| **Evidence path** | `AzureActivity` → `SigninLogs` → `AADNonInteractiveUserSignInLogs` |
| **Question** | Did a rapid multi-provider read burst originate from one Azure CLI identity and continue through background token activity? |

> **Key takeaway**  
> One resource read is normal. A fast burst across compute, storage, and network providers reveals intent to map the environment before acting.

#### Automatic correlation path

1. **Seed:** two-minute multi-provider read/list bursts expose `Caller` and `CallerIpAddress`.
2. **First join:** both fields select the Azure CLI authentication and carry `SessionId`.
3. **Final join:** the session selects background token activity and returns `Evidence`.

**Why this is defensible:** low-noise control-plane reads become meaningful when tied to the identity and continuing token that performed them.

<details><summary><strong>Supporting reference</strong></summary>

- [Azure Activity Log table](https://learn.microsoft.com/azure/azure-monitor/reference/tables/azureactivity)

</details>

In [ ]:
%%kql
let Enumeration =
    AzureActivity
    | where OperationNameValue contains "/LIST" or OperationNameValue contains "/READ"
    | summarize Operations=make_set(OperationNameValue),
                Providers=dcount(ResourceProviderValue),
                FirstSeen=min(TimeGenerated),
                LastSeen=max(TimeGenerated)
        by Caller, CallerIpAddress, WindowStart=bin(TimeGenerated, 5m)
    | where array_length(Operations) >= 3 and Providers >= 3 and LastSeen - FirstSeen <= 2m
    | project Caller, CallerIpAddress;
let Session =
    SigninLogs
    | where AppDisplayName == "Microsoft Azure CLI"
    | project Caller=UserPrincipalName, CallerIpAddress=IPAddress, SessionId
    | join kind=inner Enumeration on Caller, CallerIpAddress
    | project SessionId;
AADNonInteractiveUserSignInLogs
| project SessionId, Evidence=AuthenticationDetails
| join kind=inner Session on SessionId
| project Evidence

### APPLICATION-06 · Cloud Data Collection (Non-Mail)  
**Core scenario** · TTP ID: `cloud-data-collection-non-mail`

| Investigation lens | Detail |
|---|---|
| **ATT&CK** | [T1530 · Data from Cloud Storage](https://attack.mitre.org/techniques/T1530/) · [T1213.002 · SharePoint](https://attack.mitre.org/techniques/T1213/002/) |
| **Evidence path** | `CloudAppEvents` → `GraphAPIAuditEvents` → `OfficeActivity` |
| **Question** | Did app-only interest in a cloud file become an API retrieval and completed download? |

> **Key takeaway**  
> A cloud-app access event suggests interest. Graph and Office audit records prove the retrieval channel and final data movement.

#### Automatic correlation path

1. **Seed:** service-principal file access exposes `AppId` and `ObjectId`.
2. **First join:** Graph drive-item IDs are extracted from request URIs, then matched by both application and object ID.
3. **Final join:** the object ID selects the authoritative `FileDownloaded` audit row and returns file-name `Evidence`.

**Why this is defensible:** stable object and application identity connect cloud-app observation, Graph retrieval, and completed download without mutable names.

<details><summary><strong>Supporting reference</strong></summary>

- [Download drive-item content with Microsoft Graph](https://learn.microsoft.com/graph/api/driveitem-get-content)

</details>

In [ ]:
%%kql
let FileAccess =
    CloudAppEvents
    | where ActionType == "FileAccessed" and AccountType == "ServicePrincipal"
    | project AppId=OAuthAppId, ObjectId;
let Api =
    GraphAPIAuditEvents
    | extend ObjectId=extract(@"/drive/items/([^/]+)/content", 1, RequestUri)
    | project AppId=ApplicationId, ObjectId
    | join kind=inner FileAccess on AppId, ObjectId
    | project ObjectId;
OfficeActivity
| where Operation == "FileDownloaded"
| project ObjectId=OfficeObjectId, Evidence=SourceFileName
| join kind=inner Api on ObjectId
| project Evidence

---
## ✅ Investigation complete

All 19 hunts are self-contained, automatic correlation pipelines. Each one:

- starts from behavior-first seed evidence;
- carries correlation keys through one or two `inner` joins;
- returns only complete end-to-end matches; and
- projects the final telemetry field as `Evidence`.

### Debrief prompts

1. Which join key was most decisive: identity, source IP, session, token, policy, report, resource, or object ID?
2. Where would matching only by time or user create a false attribution?
3. Which hunt would make the strongest production analytic, and what baseline or threshold would it need?
4. Which containment action follows from the evidence: revoke sessions, remove consent, disable an app, restore Conditional Access, remove delegation, or isolate a device?

**Source:** [19 Pivot-Driven TTP Hunts](https://github.com/dcodev1702/Cyber-Defense-Workshop-ADX/blob/main/docs/ttp-hunt-queries.kql)